# Validate Paul's 52Y E-field results

Target: **Left STN** (Allen atlas acronym `STH`). Non-targets: Right STN, Left/Right M1 (`PrCG`).

Montage (from `Paul_Validation_E_Fields/52Y/README.txt`):
```
i1: AF4-FT9, 1.69 mA
i2: P9-CP2,  1.89 mA
```

Electrode positions are Paul's neuronavigation-recorded coordinates in `electrode_positions/electrode_position_52Y0X_ras`
(confirmed to match SimNIBS subject-space RAS convention against charm's own Jurak 10-10 scalp positions).

Subjects: 52Y03, 52Y04, 52Y05 (52Y02 skipped — missing `_ras` electrode file).

Run with SimNIBS's own python: `~/SimNIBS-4.6/simnibs_env/python.exe`

## Step 0: setup

In [8]:
%load_ext autoreload
%autoreload 2
# (Force reload of all modules before executing code)
import os
import numpy as np
import nibabel as nib
from scipy.ndimage import map_coordinates

PROJECT_DIR = r"D:\MINDS_Project_Karim\BIDS_TI_Toolbox"
ATLAS_DIR   = os.path.join(PROJECT_DIR, "Paul_Validation_E_Fields", "52Y", "atlas")
ELEC_DIR    = os.path.join(PROJECT_DIR, "Paul_Validation_E_Fields", "52Y", "electrode_positions")
SIMNIBS_DIR = os.path.join(PROJECT_DIR, "derivatives", "SimNIBS")

SUBJECTS = ["52Y03", "52Y04", "52Y05"]

# Allen-atlas lh/rh ROI masks Paul provided (already binary, hemisphere-split)
ROIS = {
    "STN_L": os.path.join(ATLAS_DIR, "lh", "STH.nii.gz"),
    "STN_R": os.path.join(ATLAS_DIR, "rh", "STH.nii.gz"),
    "M1_L":  os.path.join(ATLAS_DIR, "lh", "PrCG.nii.gz"),
    "M1_R":  os.path.join(ATLAS_DIR, "rh", "PrCG.nii.gz"),
}

# Montage from README.txt
MONTAGE = {
    "ch1": {"plus": "AF4", "minus": "FT9", "current_mA": 1.69},
    "ch2": {"plus": "P9",  "minus": "CP2", "current_mA": 1.89},
}

for name, path in ROIS.items():
    assert os.path.isfile(path), f"missing: {path}"
print("All ROI mask files found.")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
All ROI mask files found.


In [ ]:
import sys
sys.path.insert(0, os.path.join(PROJECT_DIR, "code", "pipeline"))
#force the reload of the imported modules to avoid caching issues
from create_masks import MaskGeneratorEngine, AllenAtlas, BNAAtlas, SimNIBSAtlas

engine = MaskGeneratorEngine("52Y03", PROJECT_DIR)

allen_labels = engine.get_subject_space_labels(AllenAtlas())
print("Allen labels in subject space:", allen_labels.shape, allen_labels.dtype,
      "unique nonzero ids:", len(np.unique(allen_labels[allen_labels > 0])))

simnibs_labels = engine.get_subject_space_labels(SimNIBSAtlas(engine.m2m_path))
print("SimNIBS (charm) labels:", simnibs_labels.shape, "unique ids:", len(np.unique(simnibs_labels)))

engine.create_roi(AllenAtlas(), {"STN_L": 10466}, "STN_L_test").sum()
engine.create_roi(AllenAtlas(), {"HC_test_by_id_head":12171, "HC_test_by_id_body": 12173, "HC_test_by_id_tail": 12174 }, "HC_Allen_byid", hemisphere="R")
engine.create_roi(BNAAtlas(), {"HC_test_by_id_1": 216, "HC_test_by_id_2":218}, "HC_BNA_byid")


engine.create_roi(BNAAtlas(), {"Caudate_test_by_id_1": 220, "caudate_test_by_id_2":228}, "Caudate_BNA_byid")
engine.create_roi(AllenAtlas(), {"Caudate_test_by_id_head":10335, "Caudate_test_by_id_body": 10336, "Caudate_test_by_id_tail": 10337 }, "Cauadate_Allen_byid", hemisphere="R")

allen_hc = engine.create_roi(AllenAtlas(), {"head": 12171, "body": 12173, "tail": 12174}, "HC_Allen_R", hemisphere="R")
bna_hc   = engine.create_roi(BNAAtlas(), {"rostral": 216, "caudal": 218}, "HC_BNA_R")
caudate_bna = engine.create_roi(BNAAtlas(), {"Caudate_test_by_id_1": 220, "Caudate_test_by_id_2":228}, "Caudate_BNA_byid")
caudate_allen = engine.create_roi(AllenAtlas(), {"Caudate_test_by_id_head":10335, "Caudate_test_by_id_body": 10336, "Caudate_test_by_id_tail": 10337 }, "Caudate_Allen_byid", hemisphere="R")

stn_r_allen = engine.create_roi(AllenAtlas(), {"STN_test_by_id_head":10466 }, "STN_Allen_byid", hemisphere="R")
out_path = os.path.join(SIMNIBS_DIR, "sub-52Y03", "comparison", "HC_atlas_compare.msh")
engine.export_visualization_mesh({"Allen_HC_R": allen_hc, "BNA_HC_R": bna_hc, "BNA_Caudate_R": caudate_bna, "Allen_Caudate_R": caudate_allen, "Allen_STN_R": stn_r_allen}, out_path)

print()


Allen labels in subject space: (192, 240, 256) int32 unique nonzero ids: 138
SimNIBS (charm) labels: (192, 240, 256) unique ids: 56
  sub-52Y03_label-STN_L_test_mask.nii.gz: 277 voxels (277.0 mm3)


np.uint64(277)

## Step 1: warp Allen ROI masks (lh/rh STH, PrCG) into each subject's head-model space

Same mechanism as the existing `create_bna_masks.py` / `_warp_bna_atlas`: charm's
`m2m_{id}/toMNI/Conform2MNI_nonl.nii.gz` stores, for every voxel of the subject's
own "Conform" grid, the corresponding **MNI world coordinate** (shape `(X,Y,Z,3)`).
We invert the Allen-atlas-mask affine to convert those MNI coordinates into Allen
voxel indices, then nearest-neighbour sample the mask — no external tools needed.

This assumes Allen atlas space ≈ SimNIBS's MNI152 target space, which we measured
at ~0.5–1.5mm affine-only shift near STN/M1 (accepted as close enough for this
validation pass).

In [2]:
def warp_mask_to_subject(mask_path, c2m, out_shape):
    """Nearest-neighbour sample a binary MNI/Allen-space mask onto the subject
    Conform grid, using the per-voxel MNI world coordinate field c2m."""
    mask_img  = nib.load(mask_path)
    mask_data = np.asarray(mask_img.dataobj, dtype=np.float32)
    aff_inv   = np.linalg.inv(mask_img.affine)

    n = c2m.shape[0] * c2m.shape[1] * c2m.shape[2]
    mni_flat = c2m.reshape(n, 3)
    ones     = np.ones((n, 1), dtype=np.float64)
    vox_mask = (aff_inv @ np.hstack([mni_flat, ones]).T).T[:, :3]
    vox_mask = vox_mask.reshape(c2m.shape[0], c2m.shape[1], c2m.shape[2], 3)

    sx, sy, sz = out_shape
    vox_mask = vox_mask[:sx, :sy, :sz, :]
    coords = [vox_mask[..., i] for i in range(3)]
    out = map_coordinates(mask_data, coords, order=0, mode='constant', cval=0)
    return (out > 0.5).astype(np.uint8)

### Step 1a: test on a single subject (52Y03) first, before looping over all three

In [7]:
test_subject = "52Y05"
m2m_path = os.path.join(SIMNIBS_DIR, f"sub-{test_subject}", f"m2m_{test_subject}")
c2m_path = os.path.join(m2m_path, "toMNI", "Conform2MNI_nonl.nii.gz")
ref_path = os.path.join(m2m_path, "segmentation", "labeling.nii.gz")

assert os.path.isfile(c2m_path), c2m_path
assert os.path.isfile(ref_path), ref_path

c2m_img = nib.load(c2m_path)
c2m = np.asarray(c2m_img.dataobj, dtype=np.float64)
print("Conform2MNI field shape:", c2m.shape)

ref_img = nib.load(ref_path)
out_shape = ref_img.shape[:3]
print("Subject Conform grid shape:", out_shape)

test_mask = warp_mask_to_subject(ROIS["STN_L"], c2m, out_shape)
vox_vol = abs(np.linalg.det(ref_img.affine[:3, :3]))
print(f"STN_L voxels: {test_mask.sum()}  ({test_mask.sum()*vox_vol:.1f} mm3)")

Conform2MNI field shape: (192, 240, 256, 3)
Subject Conform grid shape: (192, 240, 256)
STN_L voxels: 122  (122.0 mm3)


### Step 1b: run for all subjects x all 4 ROIs, save masks to derivatives/SimNIBS/sub-{id}/roi/

In [8]:
warped_masks = {}  # {(subject, roi_name): np.ndarray}

for subject in SUBJECTS:
    m2m_path = os.path.join(SIMNIBS_DIR, f"sub-{subject}", f"m2m_{subject}")
    roi_dir  = os.path.join(SIMNIBS_DIR, f"sub-{subject}", "roi")
    os.makedirs(roi_dir, exist_ok=True)

    c2m_path = os.path.join(m2m_path, "toMNI", "Conform2MNI_nonl.nii.gz")
    ref_path = os.path.join(m2m_path, "segmentation", "labeling.nii.gz")
    c2m_img = nib.load(c2m_path)
    c2m = np.asarray(c2m_img.dataobj, dtype=np.float64)
    ref_img = nib.load(ref_path)
    out_shape = ref_img.shape[:3]
    vox_vol = abs(np.linalg.det(ref_img.affine[:3, :3]))

    print(f"=== sub-{subject} ===")
    for roi_name, mask_path in ROIS.items():
        mask = warp_mask_to_subject(mask_path, c2m, out_shape)
        warped_masks[(subject, roi_name)] = mask
        out_path = os.path.join(roi_dir, f"sub-{subject}_label-{roi_name}_mask.nii.gz")
        nib.save(nib.Nifti1Image(mask, ref_img.affine, ref_img.header), out_path)
        n = int(mask.sum())
        print(f"  {roi_name:8s} {n:6d} voxels  {n*vox_vol:8.1f} mm3  -> {out_path}")
        if n == 0:
            print(f"  WARNING: {roi_name} mask empty for sub-{subject}")

=== sub-52Y03 ===
  STN_L       134 voxels     134.0 mm3  -> D:\MINDS_Project_Karim\BIDS_TI_Toolbox\derivatives\SimNIBS\sub-52Y03\roi\sub-52Y03_label-STN_L_mask.nii.gz
  STN_R       144 voxels     144.0 mm3  -> D:\MINDS_Project_Karim\BIDS_TI_Toolbox\derivatives\SimNIBS\sub-52Y03\roi\sub-52Y03_label-STN_R_mask.nii.gz
  M1_L      18771 voxels   18771.0 mm3  -> D:\MINDS_Project_Karim\BIDS_TI_Toolbox\derivatives\SimNIBS\sub-52Y03\roi\sub-52Y03_label-M1_L_mask.nii.gz
  M1_R      17814 voxels   17814.0 mm3  -> D:\MINDS_Project_Karim\BIDS_TI_Toolbox\derivatives\SimNIBS\sub-52Y03\roi\sub-52Y03_label-M1_R_mask.nii.gz
=== sub-52Y04 ===
  STN_L       165 voxels     165.0 mm3  -> D:\MINDS_Project_Karim\BIDS_TI_Toolbox\derivatives\SimNIBS\sub-52Y04\roi\sub-52Y04_label-STN_L_mask.nii.gz
  STN_R       169 voxels     169.0 mm3  -> D:\MINDS_Project_Karim\BIDS_TI_Toolbox\derivatives\SimNIBS\sub-52Y04\roi\sub-52Y04_label-STN_R_mask.nii.gz
  M1_L      18533 voxels   18533.0 mm3  -> D:\MINDS_Project_Karim\

### Step 1c: visual sanity check — overlay each ROI mask on the subject's own T1

Confirms the warped ROI actually lands in a plausible deep-brain (STN) / precentral gyrus (M1) location,
not somewhere nonsensical — important since we accepted an approximate atlas-template alignment.

In [29]:
import matplotlib.pyplot as plt

plot_ROI = False
def plot_roi_overlay(subject, roi_name, mask):
    m2m_path = os.path.join(SIMNIBS_DIR, f"sub-{subject}", f"m2m_{subject}")
    t1_img = nib.load(os.path.join(m2m_path, "T1.nii.gz"))
    t1 = np.asarray(t1_img.dataobj)

    zs = np.where(mask.sum(axis=(0, 1)) > 0)[0]  # axial (top) extent
    ys = np.where(mask.sum(axis=(0, 2)) > 0)[0]  # coronal (frontal) extent
    if len(zs) == 0 or len(ys) == 0:
        print(f"  {subject} {roi_name}: empty mask, nothing to plot")
        return
    z_mid = int(np.round(zs.mean()))
    y_mid = int(np.round(ys.mean() +2))

    fig, axes = plt.subplots(1, 2, figsize=(8, 4))

    # Top (axial) view
    axes[0].imshow(t1[:, :, z_mid].T, cmap='gray', origin='lower')
    roi_axial = np.ma.masked_where(mask[:, :, z_mid].T == 0, mask[:, :, z_mid].T)
    axes[0].imshow(roi_axial, cmap='autumn', alpha=0.6, origin='lower')
    axes[0].set_title(f"axial (z={z_mid})")
    axes[0].axis('off')

    # Frontal (coronal) view
    axes[1].imshow(t1[:, y_mid, :].T, cmap='gray', origin='lower')
    roi_coronal = np.ma.masked_where(mask[:, y_mid, :].T == 0, mask[:, y_mid, :].T)
    axes[1].imshow(roi_coronal, cmap='autumn', alpha=0.6, origin='lower')
    axes[1].set_title(f"coronal (y={y_mid})")
    axes[1].axis('off')

    fig.suptitle(f"sub-{subject}  {roi_name}")
    plt.tight_layout()
    plt.show()




if plot_ROI:
    for subject in SUBJECTS:
        for roi_name in ROIS:
            plot_roi_overlay(subject, roi_name, warped_masks[(subject, roi_name)])

## Step 2: load electrode positions (Paul's neuronavigation `_ras` files)


In [15]:
def load_electrode_positions(subject):
    path = os.path.join(ELEC_DIR, f"electrode_position_{subject}_ras")
    coords = {}
    with open(path) as f:
        next(f)  # header: #,id,x,y,z
        for line in f:
            parts = line.strip().split(",")
            name = parts[1]
            xyz = np.array([float(v) for v in parts[2:5]])
            coords[name] = xyz
    return coords

electrode_coords = {s: load_electrode_positions(s) for s in SUBJECTS}
for s in SUBJECTS:
    print(s, "electrodes loaded:", len(electrode_coords[s]))
    for name in ["AF4", "FT9", "P9", "CP2"]:
        print(f"    {name}: {electrode_coords[s][name]}")


52Y03 electrodes loaded: 74
    AF4: [36.60824596 92.69466808 25.73324789]
    FT9: [-88.60089099  26.83733163 -31.03998672]
    P9: [-71.80743342 -48.63079574 -46.61251947]
    CP2: [ 39.29812894 -41.0692056   63.76324792]
52Y04 electrodes loaded: 74
    AF4: [50.00865583 95.3379448  40.17559828]
    FT9: [-82.86654922  46.24058598 -12.5848691 ]
    P9: [-75.09795083 -40.57171709 -49.58510988]
    CP2: [ 39.93444782 -48.19351325  81.66899257]
52Y05 electrodes loaded: 74
    AF4: [27.65179409 93.73489653 40.77471065]
    FT9: [-76.44345328  13.539101   -33.57784389]
    P9: [-55.20174238 -64.38465647 -25.73071207]
    CP2: [ 32.01880678 -22.86936689  91.82802991]


## Step 3: manual FEM simulation with exact electrode positions

Same approach as `run_manual_fem_channel` in the Pablo notebook's `manual_fem_pablo_exactpos`
variant — direct tDCS FEM per channel using exact electrode coordinates, no leadfield.
ELECTRODE_DIMS/THICKNESS match that same convention.


In [ ]:
def write_electrode_diagnostic_geo(subject, electrode_names):
    m2m_path = os.path.join(SIMNIBS_DIR, f"sub-{subject}", f"m2m_{subject}")
    out_dir  = os.path.join(SIMNIBS_DIR, f"sub-{subject}", "comparison", "electrode_diagnostic")
    os.makedirs(out_dir, exist_ok=True)

    msh = mesh_io.read_msh(os.path.join(m2m_path, f"{subject}.msh"))
    msh_scalp = msh.crop_mesh(tags=[1005])   # scalp SURFACE triangles, not volume tets
    scalp_nodes = msh_scalp.nodes.node_coord
    msh_scalp.elmdata = []
    scalp_path = os.path.join(out_dir, f"sub-{subject}_scalp.msh")
    mesh_io.write_msh(msh_scalp, scalp_path)
    print(f"scalp surface: {msh_scalp.elm.nr} triangles")

    geo_lines = [
        f'// sub-{subject} electrode diagnostic\n\n',
        f'Merge "{os.path.basename(scalp_path)}";\n\n',
    ]
    pt_id = 1
    for name in electrode_names:
        p_raw = electrode_coords[subject][name]
        d = np.linalg.norm(scalp_nodes - p_raw, axis=1)
        p_snap = scalp_nodes[d.argmin()]

        geo_lines.append(f'Point({pt_id}) = {{{p_raw[0]:.4f}, {p_raw[1]:.4f}, {p_raw[2]:.4f}, 3.0}};\n')
        geo_lines.append(f'Physical Point("{name}_raw") = {{{pt_id}}};\n')
        raw_id = pt_id; pt_id += 1

        # geo_lines.append(f'Point({pt_id}) = {{{p_snap[0]:.4f}, {p_snap[1]:.4f}, {p_snap[2]:.4f}, 3.0}};\n')
        # geo_lines.append(f'Physical Point("{name}_snapped") = {{{pt_id}}};\n')
        snap_id = pt_id; pt_id += 1

        # geo_lines.append(f'Line({pt_id}) = {{{raw_id}, {snap_id}}};\n')
        # geo_lines.append(f'Physical Line("{name}_gap_{d.min():.1f}mm") = {{{pt_id}}};\n')
        pt_id += 1

    geo_path = os.path.join(out_dir, f"sub-{subject}_electrode_diagnostic.geo")
    with open(geo_path, 'w') as f:
        f.writelines(geo_lines)
    print(f"-> {geo_path}\nOpen this in Gmsh (scalp mesh + raw/snapped points + gap lines).")

write_electrode_diagnostic_geo("52Y04", list(electrode_coords["52Y04"].keys()))


### Known limitation: electrode registration accuracy

Comparing Paul's neuronavigation-recorded positions against SimNIBS's own charm-computed
scalp positions (Jurak 10-10, subject-specific) shows a **systematic ~7.7-8.3° rotation
offset**, nearly identical across all three subjects, with ~6-7mm mean residual (up to ~17mm)
even after best-fit rigid realignment. This is likely a calibration/convention mismatch
somewhere in Paul's neuronav-to-RAS pipeline, not per-subject noise — but the root cause
wasn't identified.

**Mitigation used here:** each electrode's exact position is snapped to the nearest node on
the subject's own scalp surface before FEM simulation (same mechanism as the default
Jurak/scalp-snap path in the Pablo/73T notebook). This corrects electrode *placement* but does
not correct the underlying ~6-7mm registration error — results should be read with that
uncertainty in mind, and this should be revisited if Paul can clarify the neuronav→RAS
conversion step.


In [ ]:
import simnibs
from simnibs import sim_struct, mesh_io
from simnibs.utils import TI_utils as TI
import glob, shutil

ELECTRODE_DIMS      = [19.5, 19.5]   # mm, ellipse
ELECTRODE_THICKNESS = [4.]           # mm, single rubber layer, no gel
# FEM_SUBDIR          = "manual_fem_paul_exactpos"
FEM_SUBDIR = "manual_fem_paul_scalpsnap"

_scalp_node_cache = {}

def get_scalp_nodes(subject):
    if subject not in _scalp_node_cache:
        m2m_path = os.path.join(SIMNIBS_DIR, f"sub-{subject}", f"m2m_{subject}")
        msh = mesh_io.read_msh(os.path.join(m2m_path, f"{subject}.msh"))
        _scalp_node_cache[subject] = msh.crop_mesh(tags=[1005]).nodes.node_coord
    return _scalp_node_cache[subject]

def snap_to_scalp(subject, coord):
    nodes = get_scalp_nodes(subject)
    d = np.linalg.norm(nodes - coord, axis=1)
    return nodes[d.argmin()]

def run_manual_fem_channel(subject, plus, minus, current_mA, label, coords, force=False):
    m2m_path = os.path.join(SIMNIBS_DIR, f"sub-{subject}", f"m2m_{subject}")
    out_dir  = os.path.join(SIMNIBS_DIR, f"sub-{subject}", "comparison", FEM_SUBDIR, label)

    if force and os.path.isdir(out_dir):
        print(f"    [force] deleting {label}: {out_dir}")
        shutil.rmtree(out_dir)

    existing = sorted(glob.glob(os.path.join(out_dir, "*.msh")))
    if existing and not force:
        print(f"    [skip] {label}: {os.path.basename(existing[0])}")
        return existing[0]

    os.makedirs(out_dir, exist_ok=True)
    s = sim_struct.SESSION()
    s.subpath = m2m_path; s.pathfem = out_dir; s.open_in_gmsh = False
    tdcs = s.add_tdcslist()
    tdcs.currents = [current_mA * 1e-3, -current_mA * 1e-3]
    for nr, nm in enumerate([plus, minus], 1):
        el = tdcs.add_electrode(); el.channelnr = nr
        el.centre     = snap_to_scalp(subject, coords[nm]).tolist()
        el.shape      = 'ellipse'
        el.dimensions = ELECTRODE_DIMS
        el.thickness  = ELECTRODE_THICKNESS
    simnibs.run_simnibs(s)

    files = sorted(glob.glob(os.path.join(out_dir, "*.msh")))
    if not files:
        raise FileNotFoundError(f"FEM output not found in {out_dir}")
    return files[0]


### Step 3a: test on a single subject/channel first (52Y03, ch1) before running all 6


In [ ]:
test_subject = "52Y04"
msh1_test = run_manual_fem_channel(
    test_subject, MONTAGE["ch1"]["plus"], MONTAGE["ch1"]["minus"],
    MONTAGE["ch1"]["current_mA"], "ch1", electrode_coords[test_subject])
print("ch1 mesh:", msh1_test)


**Addendum:** ch1 (AF4–FT9) also shows an elevated current calibration error (11.68% for
sub-52Y03, just over SimNIBS's 10% warning threshold) even after scalp-snapping. Isolated to
FT9 specifically — ch2 (P9–CP2) calibrates cleanly (5.1%), so this is a local mesh-geometry
effect at that one electrode position (sharp curvature near the mastoid/hairline), not a
general snapping problem. Accepted as a bounded, documented limitation rather than pursuing
SimNIBS's `definition='conf'` electrode mode (which would require manually computing a
surface-conforming vertex outline instead of centre+dimensions).


### Step 3b: run FEM for all subjects x both channels


In [34]:
fem_meshes = {}
for subject in SUBJECTS:
    for ch_label, ch in MONTAGE.items():
        msh_path = run_manual_fem_channel(
            subject, ch["plus"], ch["minus"], ch["current_mA"],
            ch_label, electrode_coords[subject])
        fem_meshes[(subject, ch_label)] = msh_path
        print(subject, ch_label, "->", msh_path)


    [skip] ch1: 52Y03_TDCS_1_scalar.msh
52Y03 ch1 -> D:\MINDS_Project_Karim\BIDS_TI_Toolbox\derivatives\SimNIBS\sub-52Y03\comparison\manual_fem_paul_scalpsnap\ch1\52Y03_TDCS_1_scalar.msh
    [skip] ch2: 52Y03_TDCS_1_scalar.msh
52Y03 ch2 -> D:\MINDS_Project_Karim\BIDS_TI_Toolbox\derivatives\SimNIBS\sub-52Y03\comparison\manual_fem_paul_scalpsnap\ch2\52Y03_TDCS_1_scalar.msh


[ simnibs ] INFO: Head Mesh:          D:\MINDS_Project_Karim\BIDS_TI_Toolbox\derivatives\SimNIBS\sub-52Y04\m2m_52Y04\52Y04.msh
[ simnibs ] INFO: Subject Path:       D:\MINDS_Project_Karim\BIDS_TI_Toolbox\derivatives\SimNIBS\sub-52Y04\m2m_52Y04
[ simnibs ] INFO: Simulation Folder:  D:\MINDS_Project_Karim\BIDS_TI_Toolbox\derivatives\SimNIBS\sub-52Y04\comparison\manual_fem_paul_scalpsnap\ch1
[ simnibs ] INFO: Running simulations in the directory: D:\MINDS_Project_Karim\BIDS_TI_Toolbox\derivatives\SimNIBS\sub-52Y04\comparison\manual_fem_paul_scalpsnap\ch1
[ simnibs ] INFO: Running Poslist Number: 1
[ simnibs ] INFO: Began to run tDCS simulation
[ simnibs ] INFO: Channels: [1, 2]
[ simnibs ] INFO: Currents (A): [0.0016899999999999999, -0.0016899999999999999]
[ simnibs ] INFO: Placing Electrode:
definition: plane
shape: ellipse
centre: [45.996022835878, 92.595161239935, 39.3379942149768]
pos_ydir: []
dimensions: [19.5, 19.5]
thickness:[4.0]
channelnr: 1
number of holes: 0

[ simnibs ] INFO: 

52Y04 ch1 -> D:\MINDS_Project_Karim\BIDS_TI_Toolbox\derivatives\SimNIBS\sub-52Y04\comparison\manual_fem_paul_scalpsnap\ch1\52Y04_TDCS_1_scalar.msh


[ simnibs ] INFO: Running simulations in the directory: D:\MINDS_Project_Karim\BIDS_TI_Toolbox\derivatives\SimNIBS\sub-52Y04\comparison\manual_fem_paul_scalpsnap\ch2
[ simnibs ] INFO: Running Poslist Number: 1
[ simnibs ] INFO: Began to run tDCS simulation
[ simnibs ] INFO: Channels: [1, 2]
[ simnibs ] INFO: Currents (A): [0.00189, -0.00189]
[ simnibs ] INFO: Placing Electrode:
definition: plane
shape: ellipse
centre: [-65.6778346744772, -37.6265047746112, -47.9366623611064]
pos_ydir: []
dimensions: [19.5, 19.5]
thickness:[4.0]
channelnr: 1
number of holes: 0

[ simnibs ] INFO: Placing Electrode:
definition: plane
shape: ellipse
centre: [36.1973325147637, -42.7244833972083, 74.6652544691655]
pos_ydir: []
dimensions: [19.5, 19.5]
thickness:[4.0]
channelnr: 2
number of holes: 0

[ simnibs ] INFO: Using isotropic conductivities
[ simnibs ] INFO: Simulating electrode pair 2101 - 2102
[ simnibs ] INFO: Assembling FEM Matrix
[ simnibs ] INFO: 10.69 s to assemble FEM matrix
[ simnibs ] INFO: 

52Y04 ch2 -> D:\MINDS_Project_Karim\BIDS_TI_Toolbox\derivatives\SimNIBS\sub-52Y04\comparison\manual_fem_paul_scalpsnap\ch2\52Y04_TDCS_1_scalar.msh


[ simnibs ] INFO: Head Mesh:          D:\MINDS_Project_Karim\BIDS_TI_Toolbox\derivatives\SimNIBS\sub-52Y05\m2m_52Y05\52Y05.msh
[ simnibs ] INFO: Subject Path:       D:\MINDS_Project_Karim\BIDS_TI_Toolbox\derivatives\SimNIBS\sub-52Y05\m2m_52Y05
[ simnibs ] INFO: Simulation Folder:  D:\MINDS_Project_Karim\BIDS_TI_Toolbox\derivatives\SimNIBS\sub-52Y05\comparison\manual_fem_paul_scalpsnap\ch1
[ simnibs ] INFO: Running simulations in the directory: D:\MINDS_Project_Karim\BIDS_TI_Toolbox\derivatives\SimNIBS\sub-52Y05\comparison\manual_fem_paul_scalpsnap\ch1
[ simnibs ] INFO: Running Poslist Number: 1
[ simnibs ] INFO: Began to run tDCS simulation
[ simnibs ] INFO: Channels: [1, 2]
[ simnibs ] INFO: Currents (A): [0.0016899999999999999, -0.0016899999999999999]
[ simnibs ] INFO: Placing Electrode:
definition: plane
shape: ellipse
centre: [24.8281927032714, 87.3996959081843, 37.2336698168874]
pos_ydir: []
dimensions: [19.5, 19.5]
thickness:[4.0]
channelnr: 1
number of holes: 0

[ simnibs ] INFO

52Y05 ch1 -> D:\MINDS_Project_Karim\BIDS_TI_Toolbox\derivatives\SimNIBS\sub-52Y05\comparison\manual_fem_paul_scalpsnap\ch1\52Y05_TDCS_1_scalar.msh


[ simnibs ] INFO: Running simulations in the directory: D:\MINDS_Project_Karim\BIDS_TI_Toolbox\derivatives\SimNIBS\sub-52Y05\comparison\manual_fem_paul_scalpsnap\ch2
[ simnibs ] INFO: Running Poslist Number: 1
[ simnibs ] INFO: Began to run tDCS simulation
[ simnibs ] INFO: Channels: [1, 2]
[ simnibs ] INFO: Currents (A): [0.00189, -0.00189]
[ simnibs ] INFO: Placing Electrode:
definition: plane
shape: ellipse
centre: [-51.8241753315793, -62.4781653489139, -25.114074048659]
pos_ydir: []
dimensions: [19.5, 19.5]
thickness:[4.0]
channelnr: 1
number of holes: 0

[ simnibs ] INFO: Placing Electrode:
definition: plane
shape: ellipse
centre: [27.5512557867837, -21.8981426560568, 83.00736945086]
pos_ydir: []
dimensions: [19.5, 19.5]
thickness:[4.0]
channelnr: 2
number of holes: 0

[ simnibs ] INFO: Using isotropic conductivities
[ simnibs ] INFO: Simulating electrode pair 2101 - 2102
[ simnibs ] INFO: Assembling FEM Matrix
[ simnibs ] INFO: 10.36 s to assemble FEM matrix
[ simnibs ] INFO: Usi

52Y05 ch2 -> D:\MINDS_Project_Karim\BIDS_TI_Toolbox\derivatives\SimNIBS\sub-52Y05\comparison\manual_fem_paul_scalpsnap\ch2\52Y05_TDCS_1_scalar.msh



## Step 4: compute TI envelope, extract amplitude at each ROI

Same convention as the Pablo notebook: crop mesh to WM+GM+CSF on read, restrict analysis to
WM+GM [1,2], combine channel E-fields via `TI.get_maxTI`, and report the volume-weighted
99th-percentile-capped mean (`_vol_mean_capped` — the project's standard metric) plus the raw
max per ROI.


In [36]:
CACHE_TAGS = [1, 2, 3]  # WM+GM+CSF on read; filtered to WM+GM [1,2] below

def mesh_geometry_and_field(msh_path, tags=CACHE_TAGS):
    mt = mesh_io.read_msh(msh_path).crop_mesh(tags=tags)
    nodes = mt.nodes.node_coord
    conn = mt.elm.node_number_list[:, :4] - 1
    v0, v1, v2, v3 = nodes[conn[:,0]], nodes[conn[:,1]], nodes[conn[:,2]], nodes[conn[:,3]]
    centroids = nodes[conn].mean(axis=1)
    elm_volumes = np.abs(np.einsum('ni,ni->n', v1-v0, np.cross(v2-v0, v3-v0))) / 6.0
    elm_tags = mt.elm.tag1.astype(np.int32)
    ef = next(d for d in mt.elmdata if d.field_name == 'E').value
    return centroids, elm_volumes, elm_tags, ef

def mask_to_elements(mask_path, centroids):
    img = nib.load(mask_path)
    data = np.asarray(img.dataobj) > 0
    aff_inv = np.linalg.inv(img.affine)
    ones = np.ones((len(centroids), 1))
    vox = (aff_inv @ np.hstack([centroids, ones]).T).T[:, :3]
    vox_idx = np.round(vox).astype(int)
    sh = data.shape
    in_bounds = ((vox_idx[:,0]>=0)&(vox_idx[:,0]<sh[0])&
                 (vox_idx[:,1]>=0)&(vox_idx[:,1]<sh[1])&
                 (vox_idx[:,2]>=0)&(vox_idx[:,2]<sh[2]))
    mask_out = np.zeros(len(centroids), dtype=bool)
    mask_out[in_bounds] = data[vox_idx[in_bounds,0], vox_idx[in_bounds,1], vox_idx[in_bounds,2]]
    return mask_out

def vol_mean_capped(values, volumes, pct=99):
    if len(values) == 0:
        return np.nan
    cap = float(np.percentile(values, pct))
    v = np.minimum(values, cap)
    return float((v*volumes).sum() / volumes.sum())


In [37]:
results = {}  # {subject: {roi_name: {mean_V_m, max_V_m, n_elements}}}

for subject in SUBJECTS:
    centroids, elm_volumes, elm_tags, ef1 = mesh_geometry_and_field(fem_meshes[(subject, "ch1")])
    _, _, _, ef2 = mesh_geometry_and_field(fem_meshes[(subject, "ch2")])
    ti = TI.get_maxTI(ef1, ef2)

    wm_gm = np.isin(elm_tags, [1, 2])
    c_v, vol_v, ti_v = centroids[wm_gm], elm_volumes[wm_gm], ti[wm_gm]

    roi_dir = os.path.join(SIMNIBS_DIR, f"sub-{subject}", "roi")
    results[subject] = {}
    print(f"\n=== sub-{subject} ===")
    for roi_name in ROIS:
        mask_path = os.path.join(roi_dir, f"sub-{subject}_label-{roi_name}_mask.nii.gz")
        elm_mask = mask_to_elements(mask_path, c_v)
        mean_v = vol_mean_capped(ti_v[elm_mask], vol_v[elm_mask])
        max_v  = float(ti_v[elm_mask].max()) if elm_mask.any() else float('nan')
        results[subject][roi_name] = {"mean_V_m": mean_v, "max_V_m": max_v, "n_elements": int(elm_mask.sum())}
        print(f"  {roi_name:8s}  mean={mean_v:.4f} V/m  max={max_v:.4f} V/m  (n={elm_mask.sum()})")



=== sub-52Y03 ===
  STN_L     mean=0.5089 V/m  max=0.5507 V/m  (n=12)
  STN_R     mean=0.3469 V/m  max=0.4035 V/m  (n=29)
  M1_L      mean=0.3338 V/m  max=0.7922 V/m  (n=23834)
  M1_R      mean=0.2894 V/m  max=0.7163 V/m  (n=24192)

=== sub-52Y04 ===
  STN_L     mean=0.4191 V/m  max=0.4785 V/m  (n=14)
  STN_R     mean=0.3126 V/m  max=0.3780 V/m  (n=16)
  M1_L      mean=0.2860 V/m  max=0.6477 V/m  (n=19757)
  M1_R      mean=0.2795 V/m  max=0.6088 V/m  (n=22546)

=== sub-52Y05 ===
  STN_L     mean=0.4208 V/m  max=0.4656 V/m  (n=16)
  STN_R     mean=0.3108 V/m  max=0.3422 V/m  (n=12)
  M1_L      mean=0.2982 V/m  max=0.8269 V/m  (n=23899)
  M1_R      mean=0.2359 V/m  max=0.6319 V/m  (n=23067)


In [39]:
import pandas as pd

rows = []
for subject in SUBJECTS:
    row = {"subject": subject}
    for roi_name in ROIS:
        row[f"{roi_name}_mean_V/m"] = results[subject][roi_name]["mean_V_m"]
        row[f"{roi_name}_max_V/m"]  = results[subject][roi_name]["max_V_m"]
    rows.append(row)
pd.DataFrame(rows).set_index("subject")


,STN_L_mean_V/m,STN_L_max_V/m,STN_R_mean_V/m,STN_R_max_V/m,M1_L_mean_V/m,M1_L_max_V/m,M1_R_mean_V/m,M1_R_max_V/m
subject,,,,,,,,
52Y03,0.508913,0.550656,0.346862,0.403497,0.333837,0.792201,0.289389,0.716252
52Y04,0.419116,0.478515,0.312611,0.377957,0.285994,0.647680,0.279519,0.608799
52Y05,0.420825,0.465644,0.310762,0.342226,0.298244,0.826932,0.235936,0.631900
